# Notebook 21 — Multi-Input Hybrid GRU+FC

**Goal:** Train a dual-branch model with two completely separate processing streams — a GRU reading the last 10 laps as a tyre-trajectory time series, and an FC network reading 13 strategic scalars. Combine at classification. Test whether the explicit temporal/strategic architectural split produces diversity beyond NB18 LSTM (ρ=0.9126 vs MLP).

## Why a dual-branch architecture

NB18 LSTM feeds **all features** through the same sequence: LapTime, TyreLife, Cumulative_Degradation, LapTime_Delta, Position, and PitStop flow together into a single LSTM. The model must learn on its own which signals are temporal and which are strategic. NB21 **hard-codes** that separation:

| Branch | What it sees | Inductive bias |
|--------|-------------|----------------|
| GRU branch | 10-lap window of [LapTime, TyreLife, Cum_Deg, LapDelta, Position] | Tyre degradation trajectory |
| FC branch | 13 race-context scalars | Strategic pit decision context |

The two branches **never share inputs**. The GRU never sees Stint or RaceProgress; the FC branch never processes a sequence. This mirrors how an F1 strategist actually thinks: "What does the tyre trajectory look like?" is a separate question from "What is the strategic context of this race?"

## Architecture

```
GRU BRANCH
  window: (B, 10, 5)  ← 10-lap history × [LapTime, TyreLife, Cum_Deg, LapDelta, Position]
  mask:   (B, 10)     ← True = valid lap, False = left-padded zero
      → GRU(input=5, hidden=128, layers=2, dropout=0.15)
      → Last hidden state  →  (B, 128)

FC BRANCH
  strat: (B, 13)  ← [Stint, RaceProgress, laps_remaining, compound_ordinal, is_wet_tyre,
                      prime_pit_window, laps_to_driver_end, abs_position_change,
                      pos_change_rolling_std_3, PitStop_lag1, TyreLife_x_laps_remaining,
                      Degradation_x_RaceProgress, Position_x_RaceProgress]
      → Linear(13→64) + BN + ReLU + Dropout(0.2)
      → Linear(64→64) + BN + ReLU  →  (B, 64)

EMBEDDINGS (shared)
  Driver(887→32) | Race(26→8) | Compound(5→3) | Year(4→2)  →  (B, 45)

MERGE
  cat([128, 64, 45])  →  (B, 237)
      → Linear(237→128) + BN + ReLU + Dropout(0.25)
      → Linear(128→64)  + BN + ReLU + Dropout(0.1)
      → Linear(64→1)    [logit]
```

## Training
- Loss: `BCEWithLogitsLoss(pos_weight=4.03)`
- Optimizer: AdamW (lr=5e-4, wd=1e-4)
- Scheduler: CosineAnnealingLR (T_max=50)
- Batch size: 2048 · Max epochs: 80 · Early stopping patience: 10
- Lighter than NB18 (~20–30 min total on Kaggle T4)

## Phase 3 Ensemble Gate
1. **Solo AUC ≥ 0.905** — minimum strength threshold
2. **Spearman ρ < 0.90 vs MLP NB15** — architectural diversity required

**Inputs:** `train_features_v2.parquet`, `test_features_v2.parquet`, `fold_assignments.parquet`, `oof_predictions_nb15.parquet`
**Outputs:** `oof_predictions_nb21.parquet` (hybrid_oof), `test_predictions_nb21.parquet` (hybrid_pred), `models/hybrid_fold{0-4}.pt`, `models/nb21_summary.pkl`

In [ ]:
# nb21-01  Imports
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr, rankdata
import pickle, time, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = lambda x, **kw: x

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
"""
# nb21-02  Path detection + data load
import os
from pathlib import Path

if Path('/kaggle/input').exists():
    # Kaggle: processed files must be uploaded as a dataset
    # Search all /kaggle/input subdirs so the dataset name doesn't need to be hardcoded
    candidates = list(Path('/kaggle/input').rglob('train_features_v2.parquet'))
    if not candidates:
        raise FileNotFoundError(
            'Upload train_features_v2.parquet (and the other processed parquets) '
            'as a Kaggle dataset before running.'
        )
    processed_dir = candidates[0].parent
    models_dir    = Path('/kaggle/working/models')
else:
    # Local: walk up from cwd until the project root is found
    cwd = Path.cwd()
    while cwd.name != 'predict-f1-pit-stops' and cwd.parent != cwd:
        cwd = cwd.parent
    project_root  = cwd
    processed_dir = project_root / 'data' / 'processed'
    models_dir    = project_root / 'models'

models_dir.mkdir(exist_ok=True)
print(f'processed_dir: {processed_dir}')
print(f'models_dir   : {models_dir}')

train    = pd.read_parquet(processed_dir / 'train_features_v2.parquet')
test     = pd.read_parquet(processed_dir / 'test_features_v2.parquet')
folds_df = pd.read_parquet(processed_dir / 'fold_assignments.parquet')
mlp_oof  = pd.read_parquet(processed_dir / 'oof_predictions_nb15.parquet')[['id', 'mlp_oof']]

print(f'Train: {train.shape}, Test: {test.shape}')
print(f'Folds: {folds_df.fold.value_counts().sort_index().to_dict()}')

"""

# nb21-02  Path detection + data load
cwd = Path.cwd()
while cwd.name != 'predict-f1-pit-stops' and cwd.parent != cwd:
    cwd = cwd.parent
project_root = cwd
processed_dir = project_root / 'data' / 'processed'
models_dir    = project_root / 'models'
models_dir.mkdir(exist_ok=True)
print(f'Root: {project_root}')

train    = pd.read_parquet(processed_dir / 'train_features_v2.parquet')
test     = pd.read_parquet(processed_dir / 'test_features_v2.parquet')
folds_df = pd.read_parquet(processed_dir / 'fold_assignments.parquet')
mlp_oof  = pd.read_parquet(processed_dir / 'oof_predictions_nb15.parquet')[['id', 'mlp_oof']]

print(f'Train: {train.shape}, Test: {test.shape}')
print(f'Folds: {folds_df.fold.value_counts().sort_index().to_dict()}')

## Feature Sets and Hyperparameters

NB21 uses **two distinct feature sets** with completely separate roles:

**`SEQ_COLS` — 5 raw temporal columns for the 10-lap GRU window**
`[LapTime (s), TyreLife, Cumulative_Degradation_winsorized, LapTime_Delta, Position]`

Intentionally excludes `PitStop`: the GRU branch models pure tyre dynamics — how lap times degrade as tyres wear. Including `PitStop` would conflate stint-reset events with degradation trajectory. `PitStop_lag1` (just-pitted signal) goes to the strategic FC branch where it belongs.

**`STRAT_COLS` — 13 strategic scalar features for the FC branch**
A curated subset of the v2 feature set capturing race strategy context: current stint, race progress, pit window timing, position volatility, compound interactions. The FC branch never sees a sequence — it receives a single snapshot per lap.

**`pos_weight = 4.03`** = (1 − 0.199) / 0.199 — upweights pit laps (20% of data) in the BCE loss to prevent the model from ignoring the minority class.

**Window = 10 laps** (vs NB18's 20). The GRU only needs the last 10 laps to detect a tyre cliff; using fewer steps reduces memory and speeds training without losing the trajectory signal that matters (degradation accelerates in the final 5–8 laps before a cliff).

In [ ]:
# nb21-03  Settings
# Raw temporal columns for the GRU window (no PitStop — pure tyre trajectory)
SEQ_COLS = [
    'LapTime (s)', 'TyreLife', 'Cumulative_Degradation_winsorized',
    'LapTime_Delta', 'Position',
]

# Strategic scalars for the FC branch (subset of v2 feature set)
STRAT_COLS = [
    'Stint', 'RaceProgress', 'laps_remaining', 'compound_ordinal', 'is_wet_tyre',
    'prime_pit_window', 'laps_to_driver_end', 'abs_position_change',
    'pos_change_rolling_std_3', 'PitStop_lag1',
    'TyreLife_x_laps_remaining', 'Degradation_x_RaceProgress', 'Position_x_RaceProgress',
]
assert len(STRAT_COLS) == 13, f'Expected 13 strategic features, got {len(STRAT_COLS)}'

CAT_COLS   = ['Driver', 'Race', 'Compound', 'Year']
WINDOW     = 10         # 10-lap tyre trajectory window (shorter than NB18's 20)
POS_WEIGHT = 4.03       # (1-0.199)/0.199 for BCEWithLogitsLoss
BATCH_SIZE = 2048
MAX_EPOCHS = 80
PATIENCE   = 10
N_FOLDS    = 5

print(f'GRU window features : {len(SEQ_COLS)}, window size: {WINDOW}')
print(f'Strategic FC features: {len(STRAT_COLS)}')
if DEVICE.type == 'cpu':
    print('WARNING: running on CPU. Expect ~1-2 hrs for 5 folds. Upload to Kaggle T4 for ~20-30 min.')

## Entity Embeddings

Categorical columns `Driver`, `Race`, `Compound`, `Year` are encoded as learned embeddings, identical to NB15 and NB18. Label encoders are fitted on **combined train + test** so test inference never encounters an unknown index.

Embedding dimensions:
- `Driver` (887 unique) → dim 32 — driver-specific pit strategy style
- `Race` (26 unique) → dim 8 — circuit-specific pit-window timing
- `Compound` (5 unique) → dim 3 — tyre type and expected stint length
- `Year` (4 unique) → dim 2 — regulation-change year effects

The embeddings are **shared** between the two branches — they are concatenated at the merge layer, not fed into either the GRU or FC branch separately. This prevents the entity signal from being double-counted.

In [ ]:
# nb21-04  Label encoders for entity embeddings (fit on combined train+test)
combined_cats = pd.concat([train[CAT_COLS], test[CAT_COLS]], ignore_index=True)
label_encoders = {}
for col in CAT_COLS:
    le = LabelEncoder()
    le.fit(combined_cats[col].astype(str))
    label_encoders[col] = le
    print(f'  {col}: {len(le.classes_)} classes')

for col in CAT_COLS:
    le = label_encoders[col]
    train[col + '_idx'] = le.transform(train[col].astype(str))
    test[col  + '_idx'] = le.transform(test[col].astype(str))

# Embedding (vocab_size+1, emb_dim) — +1 reserves index 0 for padding
EMB_DIMS = {
    'Driver':   (len(label_encoders['Driver'].classes_)   + 1, 32),
    'Race':     (len(label_encoders['Race'].classes_)     + 1, 8),
    'Compound': (len(label_encoders['Compound'].classes_) + 1, 3),
    'Year':     (len(label_encoders['Year'].classes_)     + 1, 2),
}
print('Embedding dims:', EMB_DIMS)

## Sequence Window Construction (GRU Branch)

For each row (lap), we build a fixed 10-lap window of the **preceding laps** from the same `(Race, Year, Driver)` group — using only the 5 temporal `SEQ_COLS`.

**Algorithm** (identical to NB18, shorter window):
1. Combine train and test, sort by `(Race, Year, Driver, LapNumber)`.
2. For row `i` within a group, copy the preceding `min(i, 10)` rows into the right-aligned end of the window: `window[i, WINDOW - hist_len :]`.
3. Left-pad short windows with zeros; set `mask[i, WINDOW - hist_len :]` = True for valid laps.

**Scaling:** A `StandardScaler` is fitted on **train SEQ_COLS only**, then applied to both train and test before window construction. The strategic features (`STRAT_COLS`) are scaled separately inside each CV fold to prevent fold-level leakage.

**Key difference from NB18:** The window has 5 features (no `PitStop`). `PitStop_lag1` (whether the driver pitted on the previous lap) is a strategic signal — it goes to the FC branch, not the GRU. The GRU only sees tyre wear dynamics.

In [ ]:
# nb21-05  Build sequence windows (GRU branch: 5 temporal features, 10-lap window)
idx_cols     = ['id', 'Race', 'Year', 'Driver', 'LapNumber']
cat_idx_cols = [c + '_idx' for c in CAT_COLS]

seq_scaler = StandardScaler()
seq_scaler.fit(train[SEQ_COLS].values)

# Select all needed columns (deduplicate in case of overlap)
all_sel = list(dict.fromkeys(idx_cols + SEQ_COLS + STRAT_COLS + cat_idx_cols))

combined_df = pd.concat([
    train[all_sel + ['PitNextLap']].assign(is_train=True),
    test[ all_sel].assign(is_train=False, PitNextLap=-1),
], ignore_index=True)

combined_df = combined_df.sort_values(['Race', 'Year', 'Driver', 'LapNumber']).reset_index(drop=True)

# Normalise SEQ_COLS in place (scaler fit on train only)
combined_df[SEQ_COLS] = seq_scaler.transform(combined_df[SEQ_COLS].values).astype(np.float32)

N        = len(combined_df)
N_SF     = len(SEQ_COLS)
seq_vals = combined_df[SEQ_COLS].values.astype(np.float32)

all_windows = np.zeros((N, WINDOW, N_SF), dtype=np.float32)
all_masks   = np.zeros((N, WINDOW),       dtype=bool)

GROUP_COLS = ['Race', 'Year', 'Driver']
print(f'Building windows for {N:,} rows  (window={WINDOW}, features={N_SF})...')
t0 = time.time()

for _, grp_idx in tqdm(combined_df.groupby(GROUP_COLS, sort=False).groups.items(),
                       total=combined_df[GROUP_COLS].drop_duplicates().shape[0]):
    idxs  = grp_idx.values
    n_grp = len(idxs)
    for i in range(n_grp):
        hist_len = min(i, WINDOW)
        if hist_len > 0:
            all_windows[idxs[i], WINDOW - hist_len:] = seq_vals[idxs[i - hist_len : i]]
            all_masks[idxs[i],   WINDOW - hist_len:] = True

print(f'Done in {time.time()-t0:.1f}s')
print(f'Rows with full history (>={WINDOW} prior laps): {all_masks.all(axis=1).sum():,}')
print(f'Rows with zero history (race/stint start)     : {(~all_masks.any(axis=1)).sum():,}')

In [ ]:
# nb21-06  Split back to train / test arrays
tr_mask = combined_df['is_train'].values
te_mask = ~tr_mask

train_windows   = all_windows[tr_mask]
train_seq_masks = all_masks[tr_mask]
test_windows    = all_windows[te_mask]
test_seq_masks  = all_masks[te_mask]

# Strategic features (FC branch) — raw, scaled per fold in the CV loop
train_strat_raw = combined_df.loc[tr_mask, STRAT_COLS].values.astype(np.float32)
test_strat_raw  = combined_df.loc[te_mask, STRAT_COLS].values.astype(np.float32)

train_targets = combined_df.loc[tr_mask, 'PitNextLap'].values.astype(np.float32)
train_ids     = combined_df.loc[tr_mask, 'id'].values
test_ids      = combined_df.loc[te_mask, 'id'].values

fold_lookup  = folds_df.set_index('id')['fold']
train_folds  = fold_lookup.loc[train_ids].values

train_cat = {c: combined_df.loc[tr_mask, c+'_idx'].values for c in CAT_COLS}
test_cat  = {c: combined_df.loc[te_mask, c+'_idx'].values for c in CAT_COLS}

del all_windows, all_masks, combined_df

print(f'Train windows : {train_windows.shape},  strat: {train_strat_raw.shape}')
print(f'Test  windows : {test_windows.shape},  strat: {test_strat_raw.shape}')
print(f'Fold counts   : {pd.Series(train_folds).value_counts().sort_index().to_dict()}')

## Model Architecture

Two branches process completely disjoint inputs and merge before the classification head.

```
GRU BRANCH
  window: (B, 10, 5)  ← [LapTime, TyreLife, Cum_Deg, LapDelta, Position] — no PitStop
  mask:   (B, 10)     ← True = valid lap, False = left-padded zero
      ↓  pack_padded_sequence (skip padded steps)
  GRU(input=5, hidden=128, layers=2, dropout=0.15)
      ↓  take last-layer final hidden state h_n[-1]
  gru_feat: (B, 128)

FC STRATEGIC BRANCH
  strat: (B, 13)  ← 13 race-context scalars (StandardScaled per CV fold)
      ↓  Linear(13→64) + BatchNorm1d + ReLU + Dropout(0.2)
      ↓  Linear(64→64) + BatchNorm1d + ReLU
  strat_feat: (B, 64)

EMBEDDING BRANCH (shared)
  Driver(887→32) | Race(26→8) | Compound(5→3) | Year(4→2)
      ↓  concatenate
  emb: (B, 45)

MERGE
  cat([gru_feat, strat_feat, emb])  →  (B, 237)
      ↓  Linear(237→128) + BatchNorm1d + ReLU + Dropout(0.25)
      ↓  Linear(128→64)  + BatchNorm1d + ReLU + Dropout(0.1)
      ↓  Linear(64→1)    ← raw logit
```

**Key design choices:**
- `pack_padded_sequence` with `enforce_sorted=False` stops the GRU at each row's last valid lap, preventing zero-padded steps from corrupting the hidden state.
- The FC branch uses `BatchNorm1d` (not `LayerNorm`) because the 13 strategic features have been StandardScaled per fold — they are already zero-mean unit-variance within each batch, so BN just tracks running stats for test-time normalization.
- `Dropout(0.2)` in the FC branch is lighter than the `Dropout(0.25)` in the merge head because the strategic branch has fewer parameters to overfit.
- No attention mechanism (unlike NB19 TFT) — the GRU's last hidden state captures trajectory context implicitly, and adding attention would add complexity without clear benefit for a 10-lap window.

In [ ]:
# nb21-07  Model
class HybridGRUFC(nn.Module):
    """Dual-branch: GRU(10-lap tyre window) + FC(13 strategic scalars) + entity embeddings."""

    def __init__(self, n_strat=13, n_seq=5, gru_hidden=128, n_layers=2, gru_drop=0.15,
                 emb_dims=None):
        super().__init__()
        if emb_dims is None:
            emb_dims = EMB_DIMS

        # Entity embeddings (shared — not in either branch)
        self.driver_emb   = nn.Embedding(*emb_dims['Driver'])
        self.race_emb     = nn.Embedding(*emb_dims['Race'])
        self.compound_emb = nn.Embedding(*emb_dims['Compound'])
        self.year_emb     = nn.Embedding(*emb_dims['Year'])
        emb_total = sum(v[1] for v in emb_dims.values())  # 32+8+3+2=45

        # GRU temporal branch
        self.gru = nn.GRU(
            input_size=n_seq,
            hidden_size=gru_hidden,
            num_layers=n_layers,
            batch_first=True,
            dropout=gru_drop if n_layers > 1 else 0.0,
        )

        # FC strategic branch
        self.strat_fc = nn.Sequential(
            nn.Linear(n_strat, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
        )

        # Merge head
        merge_dim = gru_hidden + 64 + emb_total  # 128+64+45=237
        self.head = nn.Sequential(
            nn.Linear(merge_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 1),
        )

    def forward(self, window, mask, strat, driver, race, compound, year):
        emb = torch.cat([
            self.driver_emb(driver),
            self.race_emb(race),
            self.compound_emb(compound),
            self.year_emb(year),
        ], dim=1)  # (B, 45)

        # GRU branch: pack to skip padded steps
        seq_len = mask.sum(dim=1).long().clamp(min=1)
        packed  = nn.utils.rnn.pack_padded_sequence(
            window, seq_len.cpu(), batch_first=True, enforce_sorted=False
        )
        _, h_n    = self.gru(packed)
        gru_feat  = h_n[-1]  # last-layer final hidden: (B, 128)

        # FC strategic branch
        strat_feat = self.strat_fc(strat)  # (B, 64)

        merged = torch.cat([gru_feat, strat_feat, emb], dim=1)  # (B, 237)
        return self.head(merged).squeeze(1)  # (B,)


# Sanity-check param count
_m = HybridGRUFC()
n_params = sum(p.numel() for p in _m.parameters())
print(f'Model params: {n_params:,}')
del _m

In [ ]:
# nb21-08  Dataset + DataLoader
class F1HybridDataset(Dataset):
    def __init__(self, windows, masks, strat, cat_idxs, targets=None):
        self.windows  = torch.tensor(windows, dtype=torch.float32)
        self.masks    = torch.tensor(masks,   dtype=torch.bool)
        self.strat    = torch.tensor(strat,   dtype=torch.float32)
        self.driver   = torch.tensor(cat_idxs['Driver'],   dtype=torch.long)
        self.race     = torch.tensor(cat_idxs['Race'],     dtype=torch.long)
        self.compound = torch.tensor(cat_idxs['Compound'], dtype=torch.long)
        self.year     = torch.tensor(cat_idxs['Year'],     dtype=torch.long)
        self.targets  = (None if targets is None
                         else torch.tensor(targets, dtype=torch.float32))

    def __len__(self):  return len(self.windows)

    def __getitem__(self, i):
        d = dict(window=self.windows[i], mask=self.masks[i], strat=self.strat[i],
                 driver=self.driver[i], race=self.race[i],
                 compound=self.compound[i], year=self.year[i])
        if self.targets is not None:
            d['target'] = self.targets[i]
        return d


def make_loader(windows, masks, strat, cat_idxs, targets=None, shuffle=False):
    ds = F1HybridDataset(windows, masks, strat, cat_idxs, targets)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=0, pin_memory=(DEVICE.type == 'cuda'))

In [ ]:
# nb21-09  Training utilities
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    total_loss, all_logits = 0.0, []

    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for batch in loader:
            win  = batch['window'].to(DEVICE)
            mask = batch['mask'].to(DEVICE)
            st   = batch['strat'].to(DEVICE)
            drv  = batch['driver'].to(DEVICE)
            rc   = batch['race'].to(DEVICE)
            cmp  = batch['compound'].to(DEVICE)
            yr   = batch['year'].to(DEVICE)

            logits = model(win, mask, st, drv, rc, cmp, yr)

            if is_train:
                tgt  = batch['target'].to(DEVICE)
                loss = criterion(logits, tgt)
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                total_loss += loss.item() * len(win)

            all_logits.append(logits.detach().cpu().float().numpy())

    probs = torch.sigmoid(torch.tensor(np.concatenate(all_logits))).numpy()
    return probs, total_loss / max(len(loader.dataset), 1)

## 5-Fold GroupKFold Cross-Validation

Same GroupKFold by `Race + '_' + Year` as all other notebooks. For each fold:

1. **Scale strategic features** — fit a fresh `StandardScaler` on the training rows of that fold (`STRAT_COLS`), then transform validation and test rows. The sequence window was already scaled globally (fit on all train before window building) — no per-fold rescaling needed for the GRU input.
2. **Build loaders** — training loader shuffles; validation and test do not.
3. **Train** — AdamW with CosineAnnealingLR and early stopping (patience=10). Best weights are saved in CPU memory and restored before OOF generation.
4. **OOF predictions** — from best-epoch model, used for correlation analysis.
5. **Test predictions** — generated per fold; final predictions are the mean of all 5.
6. **Save fold model** — `models/hybrid_fold{k}.pt`.

In [ ]:
# nb21-10  5-fold CV training
oof_preds        = np.zeros(len(train_ids))
test_preds_folds = []
fold_aucs        = []

criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([POS_WEIGHT], device=DEVICE)
)

for fold in range(N_FOLDS):
    print(f'\n{"="*60}')
    print(f'Fold {fold}')
    print('='*60)

    tr_idx = np.where(train_folds != fold)[0]
    va_idx = np.where(train_folds == fold)[0]

    # Scale strategic features (fit on training fold only)
    strat_scaler = StandardScaler()
    tr_strat = strat_scaler.fit_transform(train_strat_raw[tr_idx])
    va_strat = strat_scaler.transform(train_strat_raw[va_idx])
    te_strat = strat_scaler.transform(test_strat_raw)

    tr_cat = {c: train_cat[c][tr_idx] for c in CAT_COLS}
    va_cat = {c: train_cat[c][va_idx] for c in CAT_COLS}

    tr_loader = make_loader(train_windows[tr_idx], train_seq_masks[tr_idx],
                            tr_strat, tr_cat,
                            targets=train_targets[tr_idx], shuffle=True)
    va_loader = make_loader(train_windows[va_idx], train_seq_masks[va_idx],
                            va_strat, va_cat)
    te_loader = make_loader(test_windows, test_seq_masks, te_strat, test_cat)

    model     = HybridGRUFC().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

    best_auc, best_epoch, patience_ctr, best_state = 0.0, 0, 0, None

    for epoch in range(MAX_EPOCHS):
        _, tr_loss   = run_epoch(model, tr_loader, criterion, optimizer)
        scheduler.step()
        val_probs, _ = run_epoch(model, va_loader, criterion)
        val_auc      = roc_auc_score(train_targets[va_idx], val_probs)

        if val_auc > best_auc:
            best_auc     = val_auc
            best_epoch   = epoch
            patience_ctr = 0
            best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_ctr += 1

        if (epoch + 1) % 10 == 0 or patience_ctr >= PATIENCE:
            print(f'  ep {epoch+1:3d}  loss={tr_loss:.4f}  val={val_auc:.4f}  '
                  f'best={best_auc:.4f} (ep{best_epoch+1})')

        if patience_ctr >= PATIENCE:
            print(f'  Early stop')
            break

    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})

    oof_probs, _ = run_epoch(model, va_loader, criterion)
    oof_preds[va_idx] = oof_probs
    fold_auc = roc_auc_score(train_targets[va_idx], oof_probs)
    fold_aucs.append(fold_auc)
    print(f'  Fold {fold} AUC: {fold_auc:.4f}')

    te_probs, _ = run_epoch(model, te_loader, criterion)
    test_preds_folds.append(te_probs)

    torch.save(best_state, models_dir / f'hybrid_fold{fold}.pt')
    print(f'  Saved hybrid_fold{fold}.pt  (best ep {best_epoch+1})')

oof_auc = roc_auc_score(train_targets, oof_preds)
print(f'\nOOF AUC : {oof_auc:.4f}')
print(f'Fold AUCs: {[f"{a:.4f}" for a in fold_aucs]}')
print(f'Fold std : {np.std(fold_aucs):.4f}')

## Correlation Analysis and Phase 3 Gate

The full Spearman ρ matrix across all Phase 3 models determines which models enter the NB20 super-ensemble. A model must pass **both gates**:
1. Solo AUC ≥ 0.905
2. ρ < 0.90 vs MLP NB15

For context, NB18 LSTM achieved ρ=0.9126 vs MLP (FAIL — too correlated). The Hybrid's explicit branch separation is expected to reduce this correlation: the GRU branch sees only 5 temporal features (no lag scalars), and the FC branch processes only 13 strategic features (no lap-time history). This asymmetric feature routing should produce a different error pattern than the MLP's smooth non-axis-aligned boundaries over all 38 features.

**Expected diversity target:** ρ ~0.80–0.84 vs MLP · ρ ~0.80–0.86 vs LSTM

The cell below loads all available Phase 3 OOF predictions and prints the full ρ matrix.

In [ ]:
# nb21-11  Correlation analysis + Phase 3 gate check
mlp_vals  = mlp_oof.set_index('id').loc[train_ids, 'mlp_oof'].values

# Load other Phase 3 OOF predictions for full rho matrix
lgbm_oof_df = pd.read_parquet(processed_dir / 'oof_predictions_nb12.parquet')
lgbm_vals   = lgbm_oof_df.set_index('id').loc[train_ids, 'lgbm_tuned_oof'].values

lstm_oof_df = pd.read_parquet(processed_dir / 'oof_predictions_nb18.parquet')
lstm_vals   = lstm_oof_df.set_index('id').loc[train_ids, 'lstm_oof'].values

# TFT excluded from NB20 (AUC gate FAIL) but shown for completeness
try:
    tft_oof_df = pd.read_parquet(processed_dir / 'oof_predictions_nb19.parquet')
    tft_vals   = tft_oof_df.set_index('id').loc[train_ids, 'tft_oof'].values
    has_tft    = True
except Exception:
    tft_vals, has_tft = None, False

hybrid_vals = oof_preds

# AUC table
models_auc = {
    'LGBM NB12':    lgbm_vals,
    'MLP NB15':     mlp_vals,
    'LSTM NB18':    lstm_vals,
    'Hybrid NB21':  hybrid_vals,
}
if has_tft:
    models_auc['TFT NB19 (excl)'] = tft_vals

print('=== Solo OOF AUCs ===')
for name, preds in models_auc.items():
    auc = roc_auc_score(train_targets, preds)
    print(f'  {name:<22} {auc:.4f}')

# Full Spearman rho matrix
print('\n=== Spearman rho matrix ===')
names = list(models_auc.keys())
preds = list(models_auc.values())
header = f'{"":>22}' + ''.join(f'{n:>18}' for n in names)
print(header)
for i, (ni, pi) in enumerate(zip(names, preds)):
    row = f'{ni:>22}'
    for j, (nj, pj) in enumerate(zip(names, preds)):
        if j < i:
            row += f'{"—":>18}'
        elif j == i:
            row += f'{"1.0000":>18}'
        else:
            rho, _ = spearmanr(pi, pj)
            row += f'{rho:>18.4f}'
    print(row)

# Gate check
rho_vs_mlp, _ = spearmanr(hybrid_vals, mlp_vals)
mlp_auc        = roc_auc_score(train_targets, mlp_vals)

print(f'\n=== Phase 3 Gate: Hybrid NB21 ===')
print(f'  Solo AUC : {oof_auc:.4f}  (threshold >= 0.905)  -> {"PASS" if oof_auc >= 0.905 else "FAIL"}')
print(f'  rho vs MLP: {rho_vs_mlp:.4f}  (threshold < 0.90)   -> {"PASS" if rho_vs_mlp < 0.90 else "FAIL"}')

# Ensemble previews
def rank_avg(*arrays):
    n = len(arrays[0])
    return sum(rankdata(a) / n for a in arrays) / len(arrays)

hybrid_rank = rankdata(hybrid_vals) / len(hybrid_vals)
mlp_rank    = rankdata(mlp_vals)    / len(mlp_vals)
lgbm_rank   = rankdata(lgbm_vals)   / len(lgbm_vals)
lstm_rank   = rankdata(lstm_vals)   / len(lstm_vals)

ens2  = roc_auc_score(train_targets, rank_avg(hybrid_vals, mlp_vals))
ens3  = roc_auc_score(train_targets, rank_avg(hybrid_vals, mlp_vals, lgbm_vals))
ens4  = roc_auc_score(train_targets, rank_avg(hybrid_vals, mlp_vals, lgbm_vals, lstm_vals))

print(f'\n=== Ensemble previews (rank avg) ===')
print(f'  Hybrid + MLP                       : {ens2:.4f}  (delta vs MLP: {ens2-mlp_auc:+.4f})')
print(f'  Hybrid + MLP + LGBM                : {ens3:.4f}  (delta vs MLP: {ens3-mlp_auc:+.4f})')
print(f'  Hybrid + MLP + LGBM + LSTM         : {ens4:.4f}  (delta vs MLP: {ens4-mlp_auc:+.4f})')

In [ ]:
# nb21-12  Average test predictions across folds
test_preds_mean = np.mean(np.stack(test_preds_folds, axis=0), axis=0)
print(f'Test preds: shape={test_preds_mean.shape}, mean={test_preds_mean.mean():.4f}, '
      f'std={test_preds_mean.std():.4f}')

In [ ]:
# nb21-13  Save outputs
oof_df = pd.DataFrame({
    'id':         train_ids,
    'fold':       train_folds,
    'PitNextLap': train_targets.astype(int),
    'hybrid_oof': oof_preds,
})
assert len(oof_df) == 439140, f'Expected 439140 rows, got {len(oof_df)}'
assert oof_df['id'].nunique() == 439140, 'Duplicate ids in OOF'
oof_df.to_parquet(processed_dir / 'oof_predictions_nb21.parquet', index=False)
print(f'Saved oof_predictions_nb21.parquet  ({len(oof_df):,} rows)')

test_out = pd.DataFrame({'id': test_ids, 'hybrid_pred': test_preds_mean})
assert len(test_out) == len(test_ids), 'Row count mismatch in test predictions'
test_out.to_parquet(processed_dir / 'test_predictions_nb21.parquet', index=False)
print(f'Saved test_predictions_nb21.parquet ({len(test_out):,} rows)')

gate_pass = bool(oof_auc >= 0.905 and rho_vs_mlp < 0.90)

summary = {
    'oof_auc':          oof_auc,
    'fold_aucs':        fold_aucs,
    'fold_std':         float(np.std(fold_aucs)),
    'rho_vs_mlp':       float(rho_vs_mlp),
    'model':            'HybridGRUFC-NB21',
    'architecture':     'GRU(5,128,2L)+FC(13->64)+Emb(45) -> head(237->128->64->1)',
    'window_size':      WINDOW,
    'gate_pass':        gate_pass,
}
with open(models_dir / 'nb21_summary.pkl', 'wb') as f:
    pickle.dump(summary, f)

print(f'\n=== NB21 Summary ===')
for k, v in summary.items():
    print(f'  {k}: {v}')

## Summary

**Artifacts written:**
- `data/processed/oof_predictions_nb21.parquet` — columns: `id`, `fold`, `PitNextLap`, `hybrid_oof`
- `data/processed/test_predictions_nb21.parquet` — columns: `id`, `hybrid_pred` (5-fold averaged)
- `models/hybrid_fold{0-4}.pt` — best-epoch PyTorch weights per fold
- `models/nb21_summary.pkl` — AUCs, ρ vs MLP, gate result, architecture string

**NB20 super-ensemble inputs (from this notebook):**
- `oof_predictions_nb21.parquet` → `hybrid_oof` column
- `test_predictions_nb21.parquet` → `hybrid_pred` column

**Gate result interpretation:**
- **Both gates pass** → Hybrid included in NB20 rank average. Target ensemble: LGBM + MLP + Hybrid (+ LSTM if ρ(LSTM,LGBM) is low enough).
- **AUC passes, ρ fails** → Outputs saved. NB20 will examine whether LSTM+Hybrid together provide net diversity relative to MLP+LGBM via the full ρ matrix.
- **AUC fails** → Hybrid excluded; NB20 falls back to MLP+LGBM (v004, LB 0.93160).

**CV→LB projection** (using confirmed +0.0166 MLP-ensemble boost):

| Scenario | Expected CV | Proj LB |
|----------|------------|---------|
| MLP + LGBM (v004 baseline) | 0.9150 | 0.9316 |
| MLP + LGBM + Hybrid | ~0.924–0.930 | ~0.940–0.947 |
| MLP + LGBM + Hybrid + LSTM | ~0.928–0.938 | ~0.945–0.955 |

**Next notebook:** NB20 — Super-ensemble + final submission. Loads all passing OOF parquets, prints the full ρ matrix, applies equal-weight rank average, and generates the final submission CSV.